In [1]:
import os
import json
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY") # service_role 권장
supabase: Client = create_client(url, key)

def run_supabase_integrated_verification():
    # 1. 아직 처리되지 않은 Pending OCR 데이터 가져오기
    pending_query = supabase.table("raw_ocr_data").select("*").eq("processing_status", "Pending").execute()
    pending_data = pending_query.data
    
    if not pending_data:
        print("💡 처리할 새로운 소명 자료가 없습니다.")
        return

    print(f"🚀 총 {len(pending_data)}건의 증빙 이관 및 검증을 시작합니다.")

    for row in pending_data:
        r_id = row['id']
        f_name = row['file_name']
        # Supabase JSONB는 파이썬에서 자동으로 dict로 변환됩니다.
        parsed = row['raw_content'] if isinstance(row['raw_content'], dict) else json.loads(row['raw_content'])
        
        customer_no = parsed['customer_number']
        reporting_date = f"{parsed['year']}-{parsed['month']}-01" # DATE 형식 규격화
        ocr_val = float(parsed['usage'])

        # [Step A] site_metric_map에서 기준 정보 조회 (사업장 ID 및 지표명)
        map_query = supabase.table("site_metric_map").select("*").eq("customer_number", customer_no).execute()
        if not map_query.data:
            print(f"⚠️ 고객번호 {customer_no}에 해당하는 매핑 정보가 없습니다.")
            continue
        
        mapping = map_query.data[0]
        site_id = mapping['site_id']
        m_name = mapping['metric_name']
        m_unit = mapping['unit']

        # [Step B] evidence_usage 테이블에 정제된 증빙 데이터 적재
        evidence_data = {
            "site_id": site_id,
            "reporting_date": reporting_date,
            "metric_name": m_name,
            "unit": m_unit,
            "ocr_value": ocr_val,
            "file_name": f_name
        }
        evidence_res = supabase.table("evidence_usage").insert(evidence_data).execute()
        evidence_id = evidence_res.data[0]['id']

        # [Step C] standard_usage에서 실적 데이터 조회 및 대조
        std_query = supabase.table("standard_usage").select("*")\
            .eq("site_id", site_id)\
            .eq("metric_name", m_name)\
            .eq("reporting_date", reporting_date).execute()

        if std_query.data:
            std_record = std_query.data[0]
            std_id = std_record['id']
            db_val = float(std_record['value'])
            
            gap = db_val - ocr_val
            abs_gap = abs(gap)

            # [Step D] 정합성 판정 로직 (팀장님 로직 이식)
            if abs_gap < 1.0:
                final_status = 5
                diag = "✅ 정합성 확인 완료"
            elif abs(db_val * 1000 - ocr_val) < 1.0 or abs(db_val / 1000 - ocr_val) < 1.0:
                final_status = 4
                diag = "⚠️ 단위 기입 오류 (kWh ↔ MWh 추정)"
            else:
                final_status = 3
                diag = f"❌ 수치 불일치 (차이: {gap:.2f})"

            # 결과 업데이트 및 로그 기록
            supabase.table("standard_usage").update({"v_status": final_status}).eq("id", std_id).execute()
            
            log_data = {
                "std_id": std_id,
                "evidence_id": evidence_id,
                "gap_value": float(gap),
                "result_code": final_status,
                "diagnosis": diag
            }
            supabase.table("verification_logs").insert(log_data).execute()
            
            print(f"[{site_id}] {reporting_date} {m_name} -> {diag}")

        # [Step E] 처리 완료 표시
        supabase.table("raw_ocr_data").update({"processing_status": "Success"}).eq("id", r_id).execute()

    print("🏁 모든 증빙 이관 및 검증 프로세스가 완료되었습니다.")

if __name__ == "__main__":
    run_supabase_integrated_verification()

🚀 총 45건의 증빙 이관 및 검증을 시작합니다.
[Site A] 2025-01-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-01-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-02-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-02-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-03-01 electricity -> ❌ 수치 불일치 (차이: -150.00)
[Site A] 2025-03-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-04-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-04-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-05-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-05-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-06-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-06-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-07-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-08-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-08-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-09-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-09-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-10-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-11-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-11-01 lng -> ✅ 정합성 확인 완료
[Site A] 2025-12-01 electricity -> ✅ 정합성 확인 완료
[Site A] 2025-12-01 lng -> ✅ 정합성 확인 완료
[Site B] 2025-01-01 ele